# Lecture 7 notebook: Comparing trend systems

Companion to Lecture 7 of *Systematic Trading Notes*. The five trend families of Lecture 6, each across its parameters,
on the 16 ETFs of Lectures 5 to 9, daily, 2000–2026, sized to 10% volatility per asset and averaged with equal weights.
The notebook answers the lecture's five questions in order: at what speed (§2), does a system add anything (§3),
what do the trades show (§4), is the difference real (§5), and will it last (§6). Costs are 5 basis points per unit
traded unless stated otherwise.

In [1]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from trading_models.data.ohlc import load_ohlc
from trading_models.plotting import MUTED, SERIES, apply_theme, save_figure

apply_theme()
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
FIG = ROOT / "lectures" / "07-comparing-trend-systems" / "figures"
TICKERS = ["SPY", "QQQ", "IWM", "EFA", "EEM", "XLE", "XLF", "XLK", "XLU", "VNQ", "AGG", "TLT", "GLD", "SLV", "GDX", "USO"]
END = "2026-09-30"                                       # fixed data end, so the tables reproduce
D = load_ohlc(TICKERS, start="1993-01-01", cache=RAW / "nb05_ohlc.csv")
C, H, L = (D[f][TICKERS].loc[:END] for f in ("close", "high", "low"))
r = np.log(C).diff()                                     # daily log returns: for signals, volatility and descriptors
rs = np.expm1(r)                                         # daily simple returns: for profit, which adds up across assets only in simple returns
vol = r.ewm(span=60, min_periods=60).std()
TARGET = 0.10 / np.sqrt(252)
START = "2000-01-01"
COST = 5 / 1e4
pd.set_option("display.width", 200)

## The five families (as in Lecture 6)

Time-series momentum (the sign of the past return), price against its simple moving average, EWMAC (a continuous
forecast), the Donchian channel (entry at the N-day high or low, exit at the N/2-day opposite channel) and a
breakout with a chandelier stop (exit three average true ranges from the best price since entry).

In [2]:
def donchian(entry):
    ex = max(entry // 2, 2)
    he, le = C.rolling(entry).max().shift(1), C.rolling(entry).min().shift(1)
    hx, lx = C.rolling(ex).max().shift(1), C.rolling(ex).min().shift(1)
    out = pd.DataFrame(0.0, index=C.index, columns=C.columns)
    for t in C.columns:
        pos, ps = 0.0, []
        for px, a, b, cx, dx in zip(C[t].to_numpy(), he[t].to_numpy(), le[t].to_numpy(), hx[t].to_numpy(), lx[t].to_numpy()):
            if np.isnan(px) or np.isnan(a):
                pos = 0.0
            elif pos == 0 and px > a:
                pos = 1.0
            elif pos == 0 and px < b:
                pos = -1.0
            elif pos == 1 and px < dx:
                pos = -1.0 if px < b else 0.0
            elif pos == -1 and px > cx:
                pos = 1.0 if px > a else 0.0
            ps.append(pos)
        out[t] = ps
    return out


tr = pd.concat({t: pd.concat([H[t] - L[t], (H[t] - C[t].shift()).abs(), (L[t] - C[t].shift()).abs()], axis=1).max(axis=1) for t in TICKERS}, axis=1)
atr = tr.rolling(20).mean()


def chandelier(entry, k=3.0):
    he, le = C.rolling(entry).max().shift(1), C.rolling(entry).min().shift(1)
    out = pd.DataFrame(0.0, index=C.index, columns=C.columns)
    for t in C.columns:
        pos, ext, ps = 0.0, np.nan, []
        for px, a, b, at in zip(C[t].to_numpy(), he[t].to_numpy(), le[t].to_numpy(), atr[t].to_numpy()):
            if np.isnan(px) or np.isnan(a) or np.isnan(at):
                pos = 0.0
            elif pos == 0:
                if px > a:
                    pos, ext = 1.0, px
                elif px < b:
                    pos, ext = -1.0, px
            elif pos == 1:
                ext = max(ext, px)
                if px < ext - k * at:
                    pos = 0.0
            else:
                ext = min(ext, px)
                if px > ext + k * at:
                    pos = 0.0
            ps.append(pos)
        out[t] = ps
    return out


def ewmac(fast):
    raw = C.ewm(span=fast).mean() - C.ewm(span=4 * fast).mean()
    f = raw / (C * vol)
    f = f / f.abs().loc[START:].median().median()
    return f.clip(-2, 2)


FAMILIES = {
    "TSMOM": {n: np.sign(np.log(C).diff(n)) for n in (10, 21, 42, 63, 126, 252, 378)},
    "Price vs SMA": {n: np.sign(C - C.rolling(n).mean()) for n in (10, 20, 50, 100, 200, 300)},
    "EWMAC": {n: ewmac(n) for n in (2, 4, 8, 16, 32, 64)},
    "Donchian": {n: donchian(n) for n in (10, 20, 40, 55, 100, 200)},
    "Chandelier": {n: chandelier(n) for n in (10, 20, 40, 55, 100, 200)},
}
FAM_COL = {f: SERIES[j] for j, f in enumerate(FAMILIES)}

V = {}          # (family, parameter) -> dict of series
for fam, d in FAMILIES.items():
    for n, sig in d.items():
        pos = (sig * TARGET / vol).loc[START:]
        gross_a = pos.shift(1) * rs.loc[START:]                     # per asset
        to_a = pos.diff().abs()
        port = gross_a.mean(axis=1)
        to = to_a.mean(axis=1)
        V[(fam, n)] = dict(sig=sig.loc[START:], pos=pos, gross_a=gross_a, port=port, to=to, net=port - to * COST)
print(len(V), "systems")

31 systems


## 2. A fair axis for speed (Table 2.1, chart F.1, chart F.2)

Three measures of speed for every system: changes of direction per asset per year; the average holding period,
from turnover (twice the average absolute position divided by the average daily change of position); and the trend
signature of Levine and Pedersen (2016), estimated here by regressing each system's signal on past returns over five
horizon buckets and summarising the weights by their average horizon.

In [3]:
BUCKETS = [(1, 5), (6, 21), (22, 63), (64, 126), (127, 252)]
past = {b: np.log(C).shift(b[0] - 1) - np.log(C).shift(b[1]) for b in BUCKETS}     # return over each past horizon
past = {b: (x / (vol * np.sqrt(b[1] - b[0] + 1))).loc[START:] for b, x in past.items()}


def signature(sig):
    X = np.column_stack([past[b].stack().reindex(sig.stack().index).values for b in BUCKETS])
    y = sig.stack().values
    ok = np.isfinite(X).all(axis=1) & np.isfinite(y)
    beta = np.linalg.lstsq(X[ok], y[ok], rcond=None)[0]
    w = np.clip(beta, 0, None); w = w / w.sum() if w.sum() > 0 else w
    mid = np.array([(a + b) / 2 for a, b in BUCKETS])
    return w, float((w * mid).sum())


def sharpe(x):
    return np.sqrt(252) * x.mean() / x.std()


rows = []
for (fam, n), v in V.items():
    s = np.sign(v["sig"]).fillna(0)
    changes = float(((s.diff().abs() > 0).sum() / 26.5).median())
    hold = float((2 * v["pos"].abs().mean() / v["pos"].diff().abs().mean()).median())
    w, avg_lag = signature(v["sig"])
    v["sigw"] = w
    rows.append(dict(family=fam, parameter=n, changes_per_year=changes, holding_days=hold, signature_lag_days=avg_lag,
                     turnover_per_year=252 * v["to"].mean(), sharpe_gross=sharpe(v["port"]), sharpe_net=sharpe(v["net"])))
T2 = pd.DataFrame(rows)
display(T2.round(2))

,family,parameter,changes_per_year,holding_days,signature_lag_days,turnover_per_year,sharpe_gross,sharpe_net
0,TSMOM,10,33.04,6.83,10.21,49.43,-0.10,-0.56
1,TSMOM,21,22.32,9.89,10.18,33.54,0.18,-0.13
2,TSMOM,42,15.58,14.04,22.01,24.16,0.40,0.17
3,TSMOM,63,11.70,17.35,27.65,19.23,0.29,0.11
4,TSMOM,126,7.70,23.34,56.00,13.83,0.53,0.40
5,TSMOM,252,5.30,32.96,102.27,9.88,0.60,0.51
6,TSMOM,378,3.92,36.29,102.14,7.95,0.45,0.38
7,Price vs SMA,10,41.98,5.44,9.88,62.26,-0.16,-0.74
8,Price vs SMA,20,27.91,7.93,10.56,41.95,0.10,-0.30
9,Price vs SMA,50,16.49,12.60,16.72,25.89,0.18,-0.07


In [4]:
# how much the gap between families depends on the axis: interpolate each family's net Sharpe ratio at common speeds
def gap_on(axis, points):
    out = []
    for x in points:
        vals = []
        for fam, g in T2.groupby("family"):
            g = g.sort_values(axis)
            if g[axis].min() <= x <= g[axis].max():
                vals.append(np.interp(np.log(x), np.log(g[axis]), g["sharpe_net"]))
        if len(vals) >= 4:
            out.append(max(vals) - min(vals))
    return np.mean(out) if out else np.nan

axes = {"changes_per_year": np.geomspace(2, 12, 6), "holding_days": np.geomspace(15, 80, 6), "signature_lag_days": np.geomspace(20, 90, 6)}
display(pd.Series({a: gap_on(a, p) for a, p in axes.items()}, name="average gap between best and worst family at equal speed").round(2))

changes_per_year      0.35
holding_days          0.30
signature_lag_days    0.09
Name: average gap between best and worst family at equal speed, dtype: float64

In [5]:
lab = {"changes_per_year": "changes of direction per asset per year", "holding_days": "average holding period, days",
       "signature_lag_days": "average horizon of the trend signature, days"}
fig = make_subplots(rows=1, cols=3, subplot_titles=[lab[a] for a in lab], horizontal_spacing=0.06)
for j, a in enumerate(lab):
    for fam, g in T2.groupby("family", sort=False):
        g = g.sort_values(a)
        fig.add_scatter(x=g[a], y=g["sharpe_net"], mode="lines+markers", name=fam, line=dict(color=FAM_COL[fam], width=1.6),
                        showlegend=(j == 0), row=1, col=j + 1)
    fig.update_xaxes(type="log", row=1, col=j + 1)
fig.update_yaxes(title_text="Sharpe ratio after 5 bp", row=1, col=1)
fig.update_layout(title="The same systems on three axes of speed (16 ETFs, 2000-2026)", height=430, width=1150, margin=dict(t=80),
                  legend=dict(orientation="h", y=-0.2, x=0))
save_figure(fig, "07_01_three_axes", FIG)
fig

In [6]:
# matched speed: for each family, the parameter whose holding period is closest to the median of all systems
target_hold = T2.holding_days.median()
MATCH = {fam: g.iloc[(np.log(g.holding_days) - np.log(target_hold)).abs().argmin()].parameter for fam, g in T2.groupby("family", sort=False)}
MATCH = {f: (int(p) if float(p).is_integer() else p) for f, p in MATCH.items()}
print("target holding period, days:", round(target_hold, 1)); print("matched systems:", MATCH)
fig = go.Figure()
names = [f"{a}-{b} days" for a, b in BUCKETS]
for fam, n in MATCH.items():
    fig.add_bar(x=names, y=V[(fam, n)]["sigw"], name=f"{fam} {n}", marker_color=FAM_COL[fam])
fig.update_layout(barmode="group", title="Trend signatures at matched speed: the weight each rule puts on returns at each past horizon",
                  yaxis_title="share of the signal's weight", height=430, width=1100, margin=dict(t=80), legend=dict(orientation="h", y=-0.2, x=0))
save_figure(fig, "07_02_signatures", FIG)
fig

target holding period, days: 24.2
matched systems: {'TSMOM': 126, 'Price vs SMA': 200, 'EWMAC': 16, 'Donchian': 40, 'Chandelier': 20}


## 3. Does a system add anything? (Table 3.1, chart F.3, chart F.4)

Correlations at matched speed; the spanning regression of each family on the other four (monthly returns, Newey–West
standard errors); and the breakeven cost, the cost per unit traded at which the Sharpe ratio falls to zero.

In [7]:
M = pd.DataFrame({f: V[(f, n)]["net"] for f, n in MATCH.items()})
corr = M.corr()
display(corr.round(2))
mon = (1 + M).resample("ME").prod() - 1


def ols_nw(y, X, lags=6):
    d = pd.concat([y, X], axis=1).dropna(); yv = d.iloc[:, 0].values; Xv = np.column_stack([np.ones(len(d)), d.iloc[:, 1:].values])
    b = np.linalg.lstsq(Xv, yv, rcond=None)[0]; e = yv - Xv @ b
    XtXi = np.linalg.inv(Xv.T @ Xv); u = Xv * e[:, None]; S = u.T @ u
    for l in range(1, lags + 1):
        G = u[l:].T @ u[:-l]; S += (1 - l / (lags + 1)) * (G + G.T)
    V_ = XtXi @ S @ XtXi
    return b[0], b[0] / np.sqrt(V_[0, 0]), 1 - e.var() / yv.var()


span = {}
for f in M:
    a, t, r2 = ols_nw(mon[f], mon.drop(columns=f))
    span[f] = {"alpha % a year": 1200 * a, "t-statistic": t, "R squared": r2, "Sharpe alone": sharpe(M[f])}
T3 = pd.DataFrame(span).T
display(T3.round(2))
T2["breakeven_bp"] = [1e4 * V[(f, p)]["port"].mean() / V[(f, p)]["to"].mean() for f, p in zip(T2.family, T2.parameter)]
display(T2[["family", "parameter", "holding_days", "breakeven_bp"]].round(1))

,TSMOM,Price vs SMA,EWMAC,Donchian,Chandelier
TSMOM,1.00,0.91,0.79,0.56,0.48
Price vs SMA,0.91,1.00,0.85,0.65,0.56
EWMAC,0.79,0.85,1.00,0.86,0.75
Donchian,0.56,0.65,0.86,1.00,0.92
Chandelier,0.48,0.56,0.75,0.92,1.00


,alpha % a year,t-statistic,R squared,Sharpe alone
TSMOM,0.74,1.97,0.77,0.40
Price vs SMA,-0.25,-0.72,0.85,0.30
EWMAC,0.23,0.64,0.91,0.24
Donchian,-0.15,-0.60,0.90,0.10
Chandelier,-0.04,-0.11,0.81,0.07


,family,parameter,holding_days,breakeven_bp
0,TSMOM,10,6.8,-1.1
1,TSMOM,21,9.9,2.9
2,TSMOM,42,14.0,8.7
3,TSMOM,63,17.3,8.0
4,TSMOM,126,23.3,20.6
5,TSMOM,252,33.0,33.3
6,TSMOM,378,36.3,32.1
7,Price vs SMA,10,5.4,-1.3
8,Price vs SMA,20,7.9,1.3
9,Price vs SMA,50,12.6,3.6


In [8]:
fig = go.Figure(go.Heatmap(z=corr.values, x=list(corr.columns), y=list(corr.index), zmin=0, zmax=1, colorscale="Blues",
                           text=corr.round(2).values, texttemplate="%{text}", showscale=False))
fig.update_layout(title="Correlation of daily returns at matched speed", height=420, width=620, margin=dict(t=70), yaxis=dict(autorange="reversed"))
save_figure(fig, "07_03_correlations", FIG)
display(fig)
fig = go.Figure()
for fam, g in T2.groupby("family", sort=False):
    g = g.sort_values("holding_days")
    fig.add_scatter(x=g.holding_days, y=g.breakeven_bp, mode="lines+markers", name=fam, line=dict(color=FAM_COL[fam]))
fig.add_hline(y=5, line=dict(color=MUTED, dash="dot"), annotation_text="5 bp, the cost assumed")
fig.update_layout(title="Breakeven cost against speed: where costs eat the edge", xaxis=dict(title="average holding period, days", type="log"),
                  yaxis=dict(title="breakeven cost, basis points per unit traded", type="log"), height=450, width=1000, margin=dict(t=80),
                  legend=dict(orientation="h", y=-0.2, x=0))
save_figure(fig, "07_04_breakeven", FIG)
fig

## 4. Trades, sides and time in the market (Tables 4.1–4.3, charts F.5–F.7)

A trade is a spell of one asset in one direction, from the day the position takes a sign to the day it changes;
its return is the sum of the daily returns of that position, in units of the 10% risk target. Shown at matched speed.

In [9]:
def trades(v):
    out = []
    for t in TICKERS:
        s = np.sign(v["pos"][t]).fillna(0).values
        g = v["gross_a"][t].fillna(0).values
        start, cur = None, 0
        for i in range(len(s)):
            if s[i] != cur:
                if cur != 0 and start is not None:
                    out.append((t, cur, i - start, g[start + 1:i + 1].sum()))
                cur, start = s[i], i
        if cur != 0 and start is not None:
            out.append((t, cur, len(s) - start, g[start + 1:].sum()))
    return pd.DataFrame(out, columns=["asset", "side", "days", "ret"])


TR = {f: trades(V[(f, n)]) for f, n in MATCH.items()}
rows = {}
for f, x in TR.items():
    win, loss = x.ret[x.ret > 0], x.ret[x.ret < 0]
    top = x.ret.sort_values(ascending=False)
    rows[f] = {"trades per asset per year": len(x) / 16 / 26.5, "average days held": x.days.mean(), "win rate %": 100 * (x.ret > 0).mean(),
               "average win / average loss": win.mean() / -loss.mean(), "skewness of trade returns": x.ret.skew(),
               "share of profit from best tenth %": 100 * top.iloc[:max(len(top) // 10, 1)].sum() / x.ret.sum()}
T41 = pd.DataFrame(rows).T
display(T41.round(2))

,trades per asset per year,average days held,win rate %,average win / average loss,skewness of trade returns,share of profit from best tenth %
TSMOM,7.94,28.57,31.41,3.21,5.54,275.34
Price vs SMA,7.78,28.95,13.94,8.63,6.00,345.43
EWMAC,4.14,55.19,18.56,6.15,4.19,306.78
Donchian,4.96,33.92,37.84,1.90,2.26,458.61
Chandelier,6.60,29.22,37.90,1.83,2.01,594.87


In [10]:
fig = make_subplots(rows=1, cols=5, subplot_titles=list(TR), shared_yaxes=True, horizontal_spacing=0.03)
for j, (f, x) in enumerate(TR.items()):
    rr = 100 * x.ret.clip(-0.15, 0.4)
    cut = 100 * x.ret.sort_values(ascending=False).iloc[len(x) // 10]
    fig.add_histogram(x=rr, nbinsx=40, marker_color=FAM_COL[f], showlegend=False, row=1, col=j + 1)
    fig.add_vline(x=cut, line=dict(color="black", dash="dot"), row=1, col=j + 1)
fig.update_layout(title="Trade returns at matched speed, % of capital at the 10% risk target (dotted: the best tenth starts here)",
                  height=380, width=1150, margin=dict(t=80))
save_figure(fig, "07_05_trade_returns", FIG)
fig

In [11]:
side = {}
for f, n in MATCH.items():
    v = V[(f, n)]; lng = (v["gross_a"].where(v["pos"].shift(1) > 0, 0)).mean(axis=1); sht = (v["gross_a"].where(v["pos"].shift(1) < 0, 0)).mean(axis=1)
    side[f] = {"long side, % a year": 100 * 252 * lng.mean(), "short side, % a year": 100 * 252 * sht.mean(),
               "Sharpe long side": sharpe(lng), "Sharpe short side": sharpe(sht)}
T42 = pd.DataFrame(side).T
display(T42.round(2))
fig = go.Figure()
fig.add_bar(x=T42.index, y=T42["long side, % a year"], name="long side", marker_color=SERIES[0])
fig.add_bar(x=T42.index, y=T42["short side, % a year"], name="short side", marker_color=SERIES[3])
fig.update_layout(barmode="group", title="Where each system earns: long and short side at matched speed, before costs", yaxis_title="% a year",
                  height=420, width=900, margin=dict(t=80), legend=dict(orientation="h", y=-0.15, x=0))
save_figure(fig, "07_06_sides", FIG)
fig

,"long side, % a year","short side, % a year",Sharpe long side,Sharpe short side
TSMOM,3.61,-0.97,0.85,-0.31
Price vs SMA,3.32,-1.19,0.79,-0.38
EWMAC,3.61,-1.33,0.65,-0.31
Donchian,2.26,-1.30,0.73,-0.41
Chandelier,2.27,-1.39,0.69,-0.41


In [12]:
# time in the market, and the Sharpe ratio while invested (per asset, median across assets)
rows = []
for (f, n), v in V.items():
    inv = v["pos"].shift(1).abs() > 0
    p = float(inv.mean().median())
    sw = np.nanmedian([sharpe(v["gross_a"][t][inv[t]]) for t in TICKERS])
    sa = np.nanmedian([sharpe(v["gross_a"][t].fillna(0)) for t in TICKERS])
    rows.append(dict(family=f, parameter=n, time_in_market=p, sharpe_asset_whole=sa, sharpe_asset_invested=sw))
T43 = pd.DataFrame(rows)
display(T43[T43.family.isin(["Donchian", "Chandelier"])].round(2))
fig = go.Figure()
for fam, g in T43.groupby("family", sort=False):
    fig.add_scatter(x=g.time_in_market, y=g.sharpe_asset_whole, mode="markers", name=f"{fam}, whole period", marker=dict(color=FAM_COL[fam], size=9))
    if fam in ("Donchian", "Chandelier"):
        fig.add_scatter(x=g.time_in_market, y=g.sharpe_asset_invested, mode="markers", name=f"{fam}, while invested",
                        marker=dict(color=FAM_COL[fam], size=9, symbol="circle-open"))
fig.update_layout(title="Sharpe ratio against time in the market (median single asset, before costs)", xaxis_title="share of days with a position",
                  yaxis_title="Sharpe ratio", height=450, width=1000, margin=dict(t=80), legend=dict(orientation="h", y=-0.2, x=0))
save_figure(fig, "07_07_time_in_market", FIG)
fig

,family,parameter,time_in_market,sharpe_asset_whole,sharpe_asset_invested
19,Donchian,10,0.67,-0.07,-0.09
20,Donchian,20,0.68,0.03,0.04
21,Donchian,40,0.68,0.09,0.11
22,Donchian,55,0.67,0.07,0.10
23,Donchian,100,0.67,0.17,0.20
24,Donchian,200,0.69,0.25,0.32
25,Chandelier,10,0.81,0.10,0.11
26,Chandelier,20,0.75,0.09,0.10
27,Chandelier,40,0.61,0.04,0.05
28,Chandelier,55,0.55,0.06,0.09


In [13]:
# marginal contribution of the system that is often flat: Chandelier on EWMAC at matched speed, flat days earning zero
a, t, r2 = ols_nw(mon["Chandelier"], mon[["EWMAC"]])
print(f"Chandelier on EWMAC: alpha {1200 * a:.2f}% a year, t {t:.2f}, R squared {r2:.2f}")

Chandelier on EWMAC: alpha -0.55% a year, t -1.12, R squared 0.58


## 4.5 Beyond the Sharpe ratio: Omega and drawdowns (Table 4.4, chart F.8)

The Omega ratio at a threshold of zero is the sum of gains divided by the sum of losses: it uses the whole
distribution of returns rather than its variance. Drawdowns measure what running a system feels like: how deep,
how long, and whether two systems lose at the same time.

In [14]:
def omega(x, thr=0.0):
    d = x - thr
    return d[d > 0].sum() / -d[d < 0].sum()


def drawdown(x):
    w = (1 + x).cumprod(); return w / w.cummax() - 1


def longest_under_water(dd):
    best = cur = 0
    for u in (dd < 0).values:
        cur = cur + 1 if u else 0; best = max(best, cur)
    return best / 252


rows = {}
DD = {}
for f in M:
    x = M[f] / M[f].std() * TARGET                                    # same 10% volatility for all
    dd = drawdown(x); DD[f] = dd
    m = (1 + x).resample("ME").prod() - 1
    rows[f] = {"Sharpe": sharpe(x), "Omega, daily": omega(x), "Omega, monthly": omega(m), "maximum drawdown %": 100 * dd.min(),
               "average drawdown %": 100 * dd.mean(), "longest under water, years": longest_under_water(dd)}
T44 = pd.DataFrame(rows).T
display(T44.round(2))
DDf = pd.DataFrame(DD)
both = pd.DataFrame({(a, b): ((DDf[a] < -0.10) & (DDf[b] < -0.10)).mean() for a, b in combinations(DDf.columns, 2)}, index=["share of days both more than 10% under water"]).T
display(pd.concat([DDf.corr().stack().rename("correlation of drawdown paths"), both.iloc[:, 0]], axis=1).dropna().round(2))
fig = go.Figure()
for f in DDf:
    fig.add_scatter(x=DDf.index, y=100 * DDf[f], name=f, line=dict(color=FAM_COL[f], width=1.2))
fig.update_layout(title="Drawdowns at matched speed and equal volatility: do the systems lose at the same time?", yaxis_title="% below the previous peak",
                  height=430, width=1100, margin=dict(t=80), legend=dict(orientation="h", y=-0.15, x=0))
save_figure(fig, "07_08_drawdowns", FIG)
fig

,Sharpe,"Omega, daily","Omega, monthly",maximum drawdown %,average drawdown %,"longest under water, years"
TSMOM,0.40,1.07,1.38,-20.50,-6.48,3.21
Price vs SMA,0.30,1.05,1.27,-17.84,-7.26,3.61
EWMAC,0.24,1.05,1.22,-22.64,-7.81,5.05
Donchian,0.10,1.02,1.08,-22.34,-8.71,6.69
Chandelier,0.07,1.01,1.06,-25.10,-9.49,6.71


correlation of drawdown paths  share of days both more than 10% under water
TSMOM        Price vs SMA                           0.80                                          0.17
             EWMAC                                  0.66                                          0.15
             Donchian                               0.36                                          0.12
             Chandelier                             0.29                                          0.14
Price vs SMA EWMAC                                  0.71                                          0.19
             Donchian                               0.52                                          0.18
             Chandelier                             0.42                                          0.21
EWMAC        Donchian                               0.80                                          0.23
             Chandelier                             0.78                                          0.25
Donchian     Chandelier                             0.83                                          0.32

## 5. Is the difference real? (Table 5.1, Table 5.2, charts F.9, F.10)

Block-bootstrap bands: resample 21-day blocks of days, the same blocks for every system so that their correlation
is kept, and recompute every curve 500 times. A bootstrap test of the Sharpe-ratio difference for pairs at equal
speed, in the spirit of Ledoit and Wolf (2008). The probability of backtest overfitting (Bailey et al., 2017) by
combinatorially symmetric cross-validation: 16 blocks of time, every split into 8 and 8.

In [15]:
NET = pd.DataFrame({k: v["net"] for k, v in V.items()}).dropna()
X = NET.values
rng = np.random.default_rng(7)
B, BL = 500, 21
nb = int(np.ceil(len(X) / BL))
boots = np.empty((B, X.shape[1]))
for b in range(B):
    starts = rng.integers(0, len(X) - BL, nb)
    idx_ = (starts[:, None] + np.arange(BL)).ravel()[:len(X)]
    xb = X[idx_]
    boots[b] = np.sqrt(252) * xb.mean(0) / xb.std(0)
band = pd.DataFrame({"low": np.percentile(boots, 5, 0), "high": np.percentile(boots, 95, 0)}, index=NET.columns)
print("average width of the 90% band:", round(float((band.high - band.low).mean()), 2))
fig = go.Figure()
for fam, g in T2.groupby("family", sort=False):
    g = g.sort_values("holding_days"); keys = list(zip(g.family, g.parameter))
    lo, hi = band.loc[keys, "low"].values, band.loc[keys, "high"].values
    fig.add_scatter(x=list(g.holding_days) + list(g.holding_days[::-1]), y=list(hi) + list(lo[::-1]), fill="toself", line=dict(width=0),
                    fillcolor=FAM_COL[fam], opacity=0.15, showlegend=False, hoverinfo="skip")
    fig.add_scatter(x=g.holding_days, y=g.sharpe_net, mode="lines+markers", name=fam, line=dict(color=FAM_COL[fam]))
fig.update_layout(title="The curves of chart F.1 with 90% block-bootstrap bands (after 5 bp)", xaxis=dict(title="average holding period, days", type="log"),
                  yaxis_title="Sharpe ratio", height=460, width=1050, margin=dict(t=80), legend=dict(orientation="h", y=-0.2, x=0))
save_figure(fig, "07_09_bootstrap_bands", FIG)
fig

average width of the 90% band: 0.55


In [16]:
# test of a Sharpe-ratio difference at equal speed, by block bootstrap of the difference
def sharpe_diff_test(a, b, B=2000, BL=21, seed=11):
    d = pd.concat([a, b], axis=1).dropna().values; n = len(d); rng = np.random.default_rng(seed)
    obs = np.sqrt(252) * (d[:, 0].mean() / d[:, 0].std() - d[:, 1].mean() / d[:, 1].std())
    k = int(np.ceil(n / BL)); diffs = np.empty(B)
    for i in range(B):
        ix = (rng.integers(0, n - BL, k)[:, None] + np.arange(BL)).ravel()[:n]; x = d[ix]
        diffs[i] = np.sqrt(252) * (x[:, 0].mean() / x[:, 0].std() - x[:, 1].mean() / x[:, 1].std())
    p = 2 * min((diffs - obs <= -abs(obs)).mean() + (diffs - obs >= abs(obs)).mean(), 0.5) if False else 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return obs, np.percentile(diffs, 5), np.percentile(diffs, 95), p


pairs = [("EWMAC", "TSMOM"), ("EWMAC", "Price vs SMA"), ("Donchian", "Chandelier"), ("EWMAC", "Donchian"), ("TSMOM", "Price vs SMA")]
T51 = pd.DataFrame({f"{a} vs {b}": dict(zip(["difference", "5%", "95%", "p-value"], sharpe_diff_test(V[(a, MATCH[a])]["net"], V[(b, MATCH[b])]["net"])))
                    for a, b in pairs}).T
display(T51.round(2))

,difference,5%,95%,p-value
EWMAC vs TSMOM,-0.16,-0.34,0.04,0.18
EWMAC vs Price vs SMA,-0.05,-0.21,0.11,0.62
Donchian vs Chandelier,0.03,-0.10,0.14,0.77
EWMAC vs Donchian,0.14,0.01,0.29,0.08
TSMOM vs Price vs SMA,0.10,-0.03,0.24,0.21


In [17]:
# probability of backtest overfitting by combinatorially symmetric cross-validation (16 blocks of time)
def pbo(R, S=16):
    n = len(R) // S * S; blocks = np.array_split(np.arange(n), S); Rv = R.values[:n]
    mu = np.array([Rv[b].mean(0) for b in blocks]); sd = np.array([Rv[b].std(0) for b in blocks])
    sums, sq, cnt = mu * np.array([len(b) for b in blocks])[:, None], None, None
    logits, perf_oos, perf_is = [], [], []
    lens = np.array([len(b) for b in blocks])[:, None]
    m2 = np.array([(Rv[b] ** 2).mean(0) for b in blocks])
    for c in combinations(range(S), S // 2):
        i = np.zeros(S, bool); i[list(c)] = True
        def sr(mask):
            w = lens[mask]; m = (mu[mask] * w).sum(0) / w.sum(); v = (m2[mask] * w).sum(0) / w.sum() - m ** 2
            return m / np.sqrt(v)
        is_, oos = sr(i), sr(~i)
        best = is_.argmax()
        rank = (oos < oos[best]).mean() + 0.5 / len(oos)           # relative rank out of sample, in (0, 1)
        logits.append(np.log(rank / (1 - rank))); perf_is.append(is_[best]); perf_oos.append(oos[best])
    logits = np.array(logits)
    return (logits <= 0).mean(), np.sqrt(252) * np.median(perf_oos), np.sqrt(252) * np.median(perf_is)


rows = {}
for fam in FAMILIES:
    cols = [k for k in NET.columns if k[0] == fam]
    rows[fam] = dict(zip(["probability of overfitting", "best in sample, median Sharpe out of sample", "best in sample, median Sharpe in sample"], pbo(NET[cols])))
rows["all 31 systems"] = dict(zip(["probability of overfitting", "best in sample, median Sharpe out of sample", "best in sample, median Sharpe in sample"], pbo(NET)))
T52 = pd.DataFrame(rows).T
display(T52.round(2))

,probability of overfitting,"best in sample, median Sharpe out of sample","best in sample, median Sharpe in sample"
TSMOM,0.13,0.39,0.54
Price vs SMA,0.00,0.40,0.42
EWMAC,0.00,0.51,0.53
Donchian,0.00,0.44,0.44
Chandelier,0.07,0.24,0.27
all 31 systems,0.06,0.41,0.57


In [18]:
# choose on half of the assets, test on the other half, over the same period: one asset per pair in each half
HALF_A = ["SPY", "EFA", "XLE", "XLF", "XLU", "AGG", "GLD", "GDX"]
HALF_B = ["QQQ", "IWM", "EEM", "XLK", "VNQ", "TLT", "SLV", "USO"]
rows = []
for (f, n), v in V.items():
    na = v["gross_a"][HALF_A].mean(axis=1) - v["to"].mul(0) - (v["pos"][HALF_A].diff().abs().mean(axis=1) * COST)
    nb_ = v["gross_a"][HALF_B].mean(axis=1) - (v["pos"][HALF_B].diff().abs().mean(axis=1) * COST)
    rows.append(dict(family=f, parameter=n, sharpe_A=sharpe(na), sharpe_B=sharpe(nb_)))
TS = pd.DataFrame(rows)
TS["rank_A"] = TS.groupby("family").sharpe_A.rank(); TS["rank_B"] = TS.groupby("family").sharpe_B.rank()
rho = TS.groupby("family").apply(lambda g: g.rank_A.corr(g.rank_B), include_groups=False)
best_A = TS.loc[TS.groupby("family").sharpe_A.idxmax()].set_index("family")
med_B = TS.groupby("family").sharpe_B.median()
display(pd.DataFrame({"rank correlation, half A against half B": rho, "Sharpe on B of the best on A": best_A.sharpe_B,
                      "median Sharpe on B": med_B, "best on B": TS.groupby("family").sharpe_B.max()}).round(2))
print("rank correlation across all 31 systems:", round(TS.sharpe_A.rank().corr(TS.sharpe_B.rank()), 2))
fig = go.Figure()
for fam, g in TS.groupby("family", sort=False):
    fig.add_scatter(x=g.sharpe_A, y=g.sharpe_B, mode="markers+text", text=[str(p) for p in g.parameter], textposition="top center",
                    name=fam, marker=dict(color=FAM_COL[fam], size=9))
lim = [min(TS.sharpe_A.min(), TS.sharpe_B.min()) - 0.05, max(TS.sharpe_A.max(), TS.sharpe_B.max()) + 0.05]
fig.add_scatter(x=lim, y=lim, mode="lines", line=dict(color=MUTED, dash="dot"), showlegend=False)
fig.update_layout(title="Each system on one half of the assets against the other half, same period (after 5 bp)",
                  xaxis_title="Sharpe ratio, half A", yaxis_title="Sharpe ratio, half B", height=520, width=900, margin=dict(t=80),
                  legend=dict(orientation="h", y=-0.15, x=0))
save_figure(fig, "07_10_asset_split", FIG)
fig

,"rank correlation, half A against half B",Sharpe on B of the best on A,median Sharpe on B,best on B
family,,,,
Chandelier,0.43,0.26,0.20,0.26
Donchian,0.83,0.32,0.19,0.32
EWMAC,1.00,0.35,0.12,0.35
Price vs SMA,1.00,0.31,0.11,0.31
TSMOM,0.89,0.36,0.21,0.36


rank correlation across all 31 systems: 0.89


## 5.2 Luck or an edge? (Table 5.3, charts F.13, F.14, F.15)

1,000 strategies with no edge, built like the trend systems: in every asset a random long or short position that flips at random times, with an average holding period drawn from the 31 systems' own, the same volatility target and the same cost. The tests were fixed before any result: the luck cloud and the best of 31; persistence over time (2000-2012 against 2013-2026) and across markets (the asset halves of §5); breadth; the mechanism (profit against the size of the move, by choppiness of the asset-year, by one-month persistence) and regimes (zero rates as in Lecture 5, SPY against its 200-day average, SPY's volatility in thirds). `N_RANDOM = 200` gives a quick version (a few minutes); the lecture uses 1,000.

In [19]:
N_RANDOM = 1000                                                         # zero-edge strategies; 200 for a quick run
from scipy.stats import norm, spearmanr
R_ = r.loc[START:]; RS_ = rs.loc[START:]; idx = R_.index; HALF = idx <= pd.Timestamp("2012-12-31")
HALF_A = ["SPY", "EFA", "XLE", "XLF", "XLU", "AGG", "GLD", "GDX"]; HALF_B = ["QQQ", "IWM", "EEM", "XLK", "VNQ", "TLT", "SLV", "USO"]


def sr(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]; return float(np.sqrt(252) * x.mean() / x.std()) if len(x) > 2 and x.std() > 0 else np.nan


def books(sig):
    """As in the systems above: position = signal x target / volatility; per-asset net return; the portfolio is the mean over assets."""
    pos = (sig * TARGET / vol).loc[START:]; gross_a = pos.shift(1) * RS_; net_a = gross_a - pos.diff().abs() * COST
    return net_a, net_a.mean(axis=1), gross_a


def holding(sig):
    s = np.sign(sig.loc[START:]).fillna(0).to_numpy(); runs = [np.diff(np.flatnonzero(np.diff(s[:, j]) != 0)).mean() for j in range(s.shape[1]) if (np.diff(s[:, j]) != 0).sum() > 1]
    return float(np.mean(runs))


# descriptors of asset-years and months, fixed before any strategy is scored
yr = idx.year; ER, VR = {}, {}
for t in TICKERS:
    for y in np.unique(yr):
        d = R_[t][yr == y].dropna()
        if len(d) < 200: continue
        ER[(t, y)] = abs(d.sum()) / d.abs().sum(); m21 = d.rolling(21).sum().dropna(); VR[(t, y)] = m21.var() / (21 * d.var())
erq, vrq = np.quantile(list(ER.values()), [1 / 3, 2 / 3]), np.quantile(list(VR.values()), [1 / 3, 2 / 3])
mvol = (vol.loc[START:] * np.sqrt(21)).resample("ME").last().shift(1); zmove = (R_.resample("ME").sum() / mvol).abs()
spy = C["SPY"]; bear = (spy < spy.rolling(200).mean()).resample("ME").first().reindex(zmove.index).fillna(False)
spyvol = (r["SPY"].rolling(63).std() * np.sqrt(252)).resample("ME").first().reindex(zmove.index); vq = np.nanquantile(spyvol, [1 / 3, 2 / 3])
zero = pd.Series(False, index=zmove.index); zero[(zero.index >= "2008-12-01") & (zero.index <= "2015-12-31")] = True; zero[(zero.index >= "2020-03-01") & (zero.index <= "2022-02-28")] = True


def profile(net_a, port, gross_a):
    gp = gross_a.mean(axis=1)
    o = dict(sharpe=sr(port), g_sharpe=sr(gp), g_first=sr(gp[HALF]), g_second=sr(gp[~HALF]), g_A=sr(gross_a[HALF_A].mean(axis=1)), g_B=sr(gross_a[HALF_B].mean(axis=1)),
             g_breadth=float(np.mean([sr(gross_a[t]) > 0 for t in TICKERS])))
    py = net_a.groupby(yr).sum() * 252 / net_a.groupby(yr).count() / 0.10
    thirds = lambda D, q: (lambda a: [float(np.nanmean(a[np.digitize(a[:, 0], q) == k, 1])) for k in range(3)])(np.array([(v, py.loc[y, t]) for (t, y), v in D.items()]))
    o["er3"], o["vr3"] = thirds(ER, erq), thirds(VR, vrq)
    nm = net_a.resample("ME").sum(); xs, ys = zmove.reindex(nm.index).to_numpy().ravel(), nm.to_numpy().ravel(); ok = np.isfinite(xs) & np.isfinite(ys)
    gi = np.digitize(xs[ok], [0, 0.5, 1, 1.5, 2, 3, 99]) - 1; o["smile"] = [float(ys[ok][gi == k].mean() * 12 / 0.10) for k in range(6)]
    pm = port.resample("ME").sum()
    def srm(mask): v = pm[mask.reindex(pm.index).fillna(False).astype(bool)]; return float(np.sqrt(12) * v.mean() / v.std())
    o["regime"] = dict(zero=srm(zero), nonzero=srm(~zero), bear=srm(bear), bull=srm(~bear), vol_low=srm(spyvol < vq[0]), vol_mid=srm((spyvol >= vq[0]) & (spyvol < vq[1])), vol_high=srm(spyvol >= vq[1]))
    return o


TREND = {}
for fam, d in FAMILIES.items():
    for n, sig in d.items():
        TREND[(fam, n)] = profile(*books(sig)); TREND[(fam, n)]["hold"] = holding(sig)
holds = [v["hold"] for v in TREND.values()]; rng = np.random.default_rng(2026); RAND = []; valid = vol.notna() & C.notna()
for i in range(N_RANDOM):
    m = float(rng.choice(holds)); flips = rng.random(C.shape) < 1 / m
    s = np.where(np.cumsum(flips, axis=0) % 2 == 0, 1.0, -1.0) * np.where(rng.random(C.shape[1]) < 0.5, 1, -1)[None, :]
    sig = pd.DataFrame(s, index=C.index, columns=C.columns).where(valid); o = profile(*books(sig)); o["hold"] = holding(sig); RAND.append(o)

g = np.array([x["g_sharpe"] for x in RAND]); tg = np.array([t["g_sharpe"] for t in TREND.values()]); rng2 = np.random.default_rng(1)
best31 = np.array([g[rng2.choice(len(g), 31, replace=False)].max() for _ in range(5000)])
f1, f2 = np.array([x["g_first"] for x in RAND]), np.array([x["g_second"] for x in RAND]); win = np.argsort(-f1)[:31]
t1, t2 = np.array([t["g_first"] for t in TREND.values()]), np.array([t["g_second"] for t in TREND.values()]); tw = np.argsort(-t1)[:5]
gA, gB = np.array([x["g_A"] for x in RAND]), np.array([x["g_B"] for x in RAND]); tA, tB = np.array([t["g_A"] for t in TREND.values()]), np.array([t["g_B"] for t in TREND.values()])
T53 = pd.Series({"zero-edge: median Sharpe before costs": np.median(g), "zero-edge: spread": g.std(), "zero-edge: best of all": g.max(), "trend: best system": tg.max(), "trend: median": np.median(tg),
                 "best of 31 zero-edge: median": np.median(best31), "best of 31 zero-edge: share above the best trend system": np.mean(best31 >= tg.max()),
                 "expected maximum, 31": np.median(g) + g.std() * norm.ppf((31 - 0.375) / 31.25), "expected maximum, 1,000": np.median(g) + g.std() * norm.ppf((1000 - 0.375) / 1000.25),   # Blom's approximation to the mean of the largest of N normal draws
                 "time: correlation, zero-edge": np.corrcoef(f1, f2)[0, 1], "time: correlation, trend": np.corrcoef(t1, t2)[0, 1],
                 "time: best 31 zero-edge, first half": f1[win].mean(), "time: best 31 zero-edge, second half": f2[win].mean(), "time: best 5 trend, first half": t1[tw].mean(), "time: best 5 trend, second half": t2[tw].mean(),
                 "markets: rank correlation, zero-edge": spearmanr(gA, gB)[0], "markets: rank correlation, trend": spearmanr(tA, tB)[0]})
display(T53.round(2))

zero-edge: median Sharpe before costs                      0.01
zero-edge: spread                                          0.19
zero-edge: best of all                                     0.69
trend: best system                                         0.60
trend: median                                              0.19
best of 31 zero-edge: median                               0.40
best of 31 zero-edge: share above the best trend system    0.06
expected maximum, 31                                       0.40
expected maximum, 1,000                                    0.62
time: correlation, zero-edge                               0.02
time: correlation, trend                                   0.70
time: best 31 zero-edge, first half                        0.60
time: best 31 zero-edge, second half                      -0.08
time: best 5 trend, first half                             0.47
time: best 5 trend, second half                            0.62
markets: rank correlation, zero-edge    

In [20]:
FAMS = list(FAMILIES); COL = {f: SERIES[j] for j, f in enumerate(FAMS)}; hold_r = np.array([x["hold"] for x in RAND]); net_r = np.array([x["sharpe"] for x in RAND]); tpeak = tg.max()
fig = make_subplots(rows=1, cols=3, column_widths=[0.34, 0.3, 0.36], horizontal_spacing=0.08, subplot_titles=("1,000 strategies with no edge, before costs", "The best of 31, by luck alone", "Net of costs, against speed"))
fig.add_histogram(x=g, nbinsx=40, marker_color=MUTED, opacity=0.7, name="zero-edge strategies", row=1, col=1)
for f in FAMS:
    fig.add_scatter(x=[TREND[k]["g_sharpe"] for k in TREND if k[0] == f], y=[-4] * len(FAMILIES[f]), mode="markers", marker=dict(color=COL[f], size=9, symbol="line-ns-open", line=dict(width=2)), name=f, row=1, col=1)
fig.update_xaxes(title_text="Sharpe ratio before costs, 2000-2026", row=1, col=1); fig.update_yaxes(title_text="number of strategies", row=1, col=1)
fig.add_histogram(x=best31, nbinsx=40, marker_color=MUTED, opacity=0.7, showlegend=False, row=1, col=2); fig.add_vline(x=tpeak, line=dict(color="#222", width=2, dash="dot"), row=1, col=2)
fig.add_annotation(x=tpeak, y=1, yref="y2 domain", text=f"best trend system {tpeak:.2f}", xanchor="left", showarrow=False, row=1, col=2)
fig.update_xaxes(title_text="best Sharpe among 31 zero-edge strategies", row=1, col=2); fig.update_yaxes(title_text="number of draws (of 5,000)", row=1, col=2)
fig.add_scatter(x=hold_r, y=net_r, mode="markers", marker=dict(color=MUTED, size=4, opacity=0.5), showlegend=False, row=1, col=3)
for f in FAMS:
    ks = [k for k in TREND if k[0] == f]; fig.add_scatter(x=[TREND[k]["hold"] for k in ks], y=[TREND[k]["sharpe"] for k in ks], mode="lines+markers", line=dict(color=COL[f], width=2), showlegend=False, row=1, col=3)
fig.update_xaxes(title_text="average holding period, days", type="log", row=1, col=3); fig.update_yaxes(title_text="Sharpe ratio after 5 bp", row=1, col=3)
fig.update_layout(title="Luck or an edge? Zero-edge strategies built like the trend systems<br><sup>16 ETFs, daily, 2000-2026; random long and short positions with the trend systems' holding periods, the same volatility target and costs. Ticks and lines: the 31 trend systems, by family</sup>",
                  height=500, width=1500, margin=dict(t=110, l=70, r=30, b=120), legend=dict(orientation="h", y=-0.22, x=0))
save_figure(fig, "07_13_luck_cloud", FIG); display(fig)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1, subplot_titles=("Over time: does the first half predict the second?", "Across markets: does one half of the ETFs predict the other?"))
for col, (a, b, ka, kb) in enumerate(((f1, f2, "g_first", "g_second"), (gA, gB, "g_A", "g_B")), start=1):
    keep = np.ones(len(a), bool); keep[win] = False
    fig.add_scatter(x=a[keep], y=b[keep], mode="markers", marker=dict(color=MUTED, size=4, opacity=0.45), name="zero-edge strategies", showlegend=col == 1, row=1, col=col)
    fig.add_scatter(x=a[win], y=b[win], mode="markers", marker=dict(color=SERIES[1], size=7), name="the 31 luckiest in 2000-2012", showlegend=col == 1, row=1, col=col)
    for f in FAMS:
        ks = [k for k in TREND if k[0] == f]; fig.add_scatter(x=[TREND[k][ka] for k in ks], y=[TREND[k][kb] for k in ks], mode="markers", marker=dict(color=COL[f], size=9, symbol="diamond"), name=f, showlegend=col == 1, row=1, col=col)
    fig.add_hline(y=0, line=dict(color="#222", width=1, dash="dot"), row=1, col=col); fig.add_vline(x=0, line=dict(color="#222", width=1, dash="dot"), row=1, col=col)
fig.update_xaxes(title_text="Sharpe ratio before costs, 2000-2012", row=1, col=1); fig.update_yaxes(title_text="Sharpe ratio before costs, 2013-2026", row=1, col=1)
fig.update_xaxes(title_text="Sharpe ratio before costs, ETFs of half A", row=1, col=2); fig.update_yaxes(title_text="Sharpe ratio before costs, ETFs of half B", row=1, col=2)
fig.update_layout(title="Luck does not persist; an edge does<br><sup>Grey: zero-edge strategies; orange: the 31 luckiest in 2000-2012 (the same 31 in both panels); diamonds: the 31 trend systems. Before costs. Halves of the ETFs as in §5</sup>",
                  height=540, width=1400, margin=dict(t=110, l=70, r=30, b=120), legend=dict(orientation="h", y=-0.2, x=0))
save_figure(fig, "07_14_luck_persistence", FIG); display(fig)

slow = [t for t in TREND.values() if t["sharpe"] > 0.3]; lucky = [RAND[i] for i in np.argsort(-net_r)[:31]]; mean = lambda key, sel: np.nanmean(np.array([x[key] for x in sel]), axis=0)
fig = make_subplots(rows=1, cols=3, column_widths=[0.38, 0.27, 0.35], horizontal_spacing=0.08, subplot_titles=("What drives it: profit against the size of the move", "What kills it: choppy markets", "When it works: regimes"))
for sel, c, nm in ((slow, SERIES[0], f"slow trend systems ({len(slow)})"), (lucky, SERIES[1], "the 31 luckiest zero-edge strategies"), (RAND, MUTED, "all zero-edge strategies")):
    fig.add_scatter(x=["< 0.5", "0.5-1", "1-1.5", "1.5-2", "2-3", "> 3"], y=mean("smile", sel), mode="lines+markers", line=dict(color=c, width=2), marker=dict(size=8), name=nm, row=1, col=1)
    fig.add_scatter(x=["choppiest third", "middle third", "most trending third"], y=mean("er3", sel), mode="lines+markers", line=dict(color=c, width=2), marker=dict(size=8), showlegend=False, row=1, col=2)
reg = [("zero", "zero rates"), ("nonzero", "other months"), ("bear", "SPY below 200-day"), ("bull", "SPY above"), ("vol_low", "low volatility"), ("vol_mid", "middle"), ("vol_high", "high volatility")]
for sel, c in ((slow, SERIES[0]), (lucky, SERIES[1])):
    fig.add_bar(x=[b for _, b in reg], y=[float(np.nanmean([x["regime"][k] for x in sel])) for k, _ in reg], marker_color=c, showlegend=False, row=1, col=3)
for col in (1, 2, 3): fig.add_hline(y=0, line=dict(color="#222", width=1, dash="dot"), row=1, col=col)
fig.update_xaxes(title_text="the asset's move in the month, in its own volatility (absolute)", row=1, col=1); fig.update_yaxes(title_text="profit, annualised, in units of 10% risk", row=1, col=1)
fig.update_xaxes(title_text="asset-years by efficiency ratio (net move over path length)", row=1, col=2); fig.update_yaxes(title_text="profit, annualised, in units of 10% risk", row=1, col=2)
fig.update_xaxes(title_text="months by regime, fixed in advance", tickangle=-30, row=1, col=3); fig.update_yaxes(title_text="Sharpe ratio after 5 bp", row=1, col=3)
fig.update_layout(barmode="group", title="What drives trend's edge, what kills it, and when it works<br><sup>16 ETFs, 2000-2026, after 5 bp. Slow trend systems: Sharpe ratio above 0.3. Regimes by month: zero rates (Lecture 5), SPY against its 200-day average and SPY's volatility in thirds, at the start of the month</sup>",
                  height=520, width=1550, margin=dict(t=110, l=70, r=30, b=150), legend=dict(orientation="h", y=-0.3, x=0))
save_figure(fig, "07_15_luck_mechanism", FIG); display(fig)
top31 = sorted(RAND, key=lambda x: -x["g_sharpe"])[:31]
print("breadth (share of ETFs with a positive Sharpe ratio before costs): trend systems above 0.4:", round(float(np.mean([t["g_breadth"] for t in TREND.values() if t["g_sharpe"] > 0.4])), 2),
      "| best 31 zero-edge:", round(float(np.mean([x["g_breadth"] for x in top31])), 2))
print("variance-ratio thirds, slow trend:", np.round(mean("vr3", slow), 2), "| regimes, slow trend:", {k: round(float(np.nanmean([x["regime"][k] for x in slow])), 2) for k, _ in reg})

breadth (share of ETFs with a positive Sharpe ratio before costs): trend systems above 0.4: 0.89 | best 31 zero-edge: 0.7
variance-ratio thirds, slow trend: [0.55 0.15 0.1 ] | regimes, slow trend: {'zero': 0.45, 'nonzero': 0.53, 'bear': -0.19, 'bull': 0.74, 'vol_low': 0.82, 'vol_mid': 0.69, 'vol_high': -0.02}


## 6. Will it last? (Table 6.1, charts F.11, F.12)

Median against peak per family; walk-forward choice of the parameter (each year, the best on all past data, traded
the next year) against the fixed median parameter and against an equal-weighted blend of all parameters; and the
systems that no other beats on Sharpe ratio, monthly skewness and maximum drawdown at once.

In [21]:
rows = {}
years = range(2005, 2027)
for fam in FAMILIES:
    cols = [k for k in NET.columns if k[0] == fam]
    R = NET[cols]
    wf = []
    for y in years:
        past_ = R.loc[:f"{y - 1}-12-31"]
        best = past_.mean().div(past_.std()).idxmax()
        wf.append(R.loc[str(y), best])
    wf = pd.concat(wf)
    med_k = sorted(cols, key=lambda k: k[1])[len(cols) // 2]
    blend = (R / R.std()).mean(axis=1)
    sl = slice("2005", None)
    rows[fam] = {"peak": max(sharpe(R[c]) for c in cols), "median across parameters": np.median([sharpe(R[c]) for c in cols]),
                 "walk-forward choice, 2005-2026": sharpe(wf), "fixed middle parameter, 2005-2026": sharpe(R[med_k].loc[sl]),
                 "blend of all parameters, 2005-2026": sharpe(blend.loc[sl])}
T61 = pd.DataFrame(rows).T
display(T61.round(2))
fig = go.Figure()
fig.add_bar(x=T61.index, y=T61["peak"], name="peak across parameters", marker_color=MUTED)
fig.add_bar(x=T61.index, y=T61["median across parameters"], name="median across parameters", marker_color=SERIES[0])
fig.update_layout(barmode="group", title="Median against peak: how much of each family's best result is selection (after 5 bp)",
                  yaxis_title="Sharpe ratio", height=420, width=900, margin=dict(t=80), legend=dict(orientation="h", y=-0.15, x=0))
save_figure(fig, "07_11_median_peak", FIG)
display(fig)

,peak,median across parameters,"walk-forward choice, 2005-2026","fixed middle parameter, 2005-2026","blend of all parameters, 2005-2026"
TSMOM,0.51,0.17,0.51,0.07,0.19
Price vs SMA,0.42,0.01,0.38,0.11,-0.09
EWMAC,0.53,0.10,0.51,0.21,0.02
Donchian,0.44,0.09,0.46,0.09,0.08
Chandelier,0.27,0.08,0.27,0.09,0.13


In [22]:
rows = []
for k, v in V.items():
    x = v["net"] / v["net"].std() * TARGET
    m = (1 + x).resample("ME").prod() - 1
    rows.append(dict(family=k[0], parameter=k[1], sharpe=sharpe(x), skewness=m.skew(), max_dd=100 * drawdown(x).min()))
P6 = pd.DataFrame(rows)
def dominated(i):
    a = P6.iloc[i]
    return any((P6.sharpe >= a.sharpe) & (P6.skewness >= a.skewness) & (P6.max_dd >= a.max_dd) &
               ((P6.sharpe > a.sharpe) | (P6.skewness > a.skewness) | (P6.max_dd > a.max_dd)))
P6["undominated"] = [not dominated(i) for i in range(len(P6))]
display(P6[P6.undominated].round(2))
fig = go.Figure()
for fam, g in P6.groupby("family", sort=False):
    fig.add_scatter(x=g.max_dd, y=g.sharpe, mode="markers", name=fam, text=[str(p) for p in g.parameter],
                    marker=dict(color=FAM_COL[fam], size=[14 if u else 8 for u in g.undominated], line=dict(width=[2 if u else 0 for u in g.undominated], color="black")))
fig.update_layout(title="Sharpe ratio against maximum drawdown at 10% volatility; large outlined markers: no other system is better on Sharpe, skewness and drawdown at once",
                  xaxis_title="maximum drawdown, %", yaxis_title="Sharpe ratio after 5 bp", height=480, width=1100, margin=dict(t=80),
                  legend=dict(orientation="h", y=-0.15, x=0))
save_figure(fig, "07_12_tradeoffs", FIG)
fig

,family,parameter,sharpe,skewness,max_dd,undominated
5,TSMOM,252,0.51,-0.03,-23.34,True
13,EWMAC,2,-0.76,0.95,-88.61,True
16,EWMAC,16,0.24,0.67,-22.64,True
17,EWMAC,32,0.43,0.21,-16.42,True
18,EWMAC,64,0.53,-0.07,-19.82,True
24,Donchian,200,0.44,-0.09,-19.51,True
29,Chandelier,100,0.19,0.93,-21.02,True
30,Chandelier,200,0.27,0.38,-17.37,True
